In [1]:
import os
import json
from PIL import Image
from datasets import Dataset, DatasetDict, Sequence

In [ ]:
verl_home = "/home/xwchen/myverl/verl"
original_dataset_train = os.path.join(verl_home, "data/avqa/train_qa.json")
original_dataset_val = os.path.join(verl_home, "data/avqa/val_qa.json")
audio_dir = "/home/xwchen/data/VGGSound/audio/"
data_source = "AVQA"

In [ ]:
instruction_following = "Let's think step by step. Output the thinking process in <think> </think> and final answer in <answer> </answer>."

def generate_data(data_path: str):
    with open(data_path, "r", encoding="utf-8") as f:
        datas = json.load(f)
        for data in datas:
            audio_path = os.path.join(audio_dir, data["video_name"] + ".wav")
            choice_str = f"Please choose an answer from the following options: {data['multi_choice']}."
            prompt = f"{data['question_text'].replace('video', 'audio')} {choice_str} {instruction_following}"
            if os.path.exists(audio_path):
                yield {
                    "data_source": data_source,
                    "prompt": [{
                        "role": "user",
                        "content": prompt,
                    }],
                }
                # yield {
                #     "id": data["id"],
                #     "video_name": data["video_name"],
                #     "video_id": data["video_id"],
                #     "question_text": data["question_text"],
                #     "multi_choice": data["multi_choice"],
                #     "answer": data["answer"],
                #     "question_relation": data["question_relation"],
                #     "question_type": data["question_type"],
                #     "dataset_name": "AVQA",
                #     "audio_path": audio_path,
                # }

In [14]:
trainset = Dataset.from_generator(generate_data, gen_kwargs={"data_path": original_dataset_train})

In [15]:
testset = Dataset.from_generator(generate_data, gen_kwargs={"data_path": original_dataset_val})

In [16]:
dataset = DatasetDict({"train": trainset, "test": testset})

In [17]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'video_name', 'video_id', 'question_text', 'multi_choice', 'answer', 'question_relation', 'question_type', 'dataset_name', 'audio_path'],
        num_rows: 40182
    })
    test: Dataset({
        features: ['id', 'video_name', 'video_id', 'question_text', 'multi_choice', 'answer', 'question_relation', 'question_type', 'dataset_name', 'audio_path'],
        num_rows: 16798
    })
})

In [18]:
dataset.save_to_disk("/home/xwchen/data/aqa")

Saving the dataset (0/1 shards):   0%|          | 0/40182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/16798 [00:00<?, ? examples/s]

In [1]:
import datasets

In [2]:
dataset = datasets.load_from_disk("/home/xwchen/data/aqa")

In [3]:
train_dataset = dataset['train']
test_dataset = dataset['test']

In [22]:
train_dataset.to_parquet("/home/xwchen/data/aqa/train.parquet")
test_dataset.to_parquet("/home/xwchen/data/aqa/test.parquet")

Creating parquet from Arrow format:   0%|          | 0/41 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/17 [00:00<?, ?ba/s]

3874470

In [15]:
train_dataset[7]

{'id': 191,
 'video_name': '-HxQ9AoyRmY_000030',
 'video_id': 355,
 'question_text': 'What is shown in the video?',
 'multi_choice': ['A flock of chickens', 'frog', 'gibbon', 'duck'],
 'answer': 0,
 'question_relation': 'Both',
 'question_type': 'Which',
 'dataset_name': 'AVQA',
 'audio_path': '/home/xwchen/data/VGGSound/audio/-HxQ9AoyRmY_000030.wav'}